# 05 Build Ablation Datasets

Build YOLO-ready dataset folders for the four ablation experiments.


## Purpose

This notebook assembles clean experiment folders from the original split produced by Notebook 03 and the offline augmented training data produced by Notebook 04.

Validation and test sets are copied identically into every experiment. Only the training split changes between experiments, which keeps later ablation comparisons fair.


## Setup

Find the `v2_pipeline` root, add it to `sys.path`, and import reusable dataset-building helpers from `src/`.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display


def find_v2_root(start: Path) -> Path:
    """Find the v2_pipeline root from the current working directory."""
    for candidate in (start, *start.parents):
        if candidate.name == "v2_pipeline" and (candidate / "src").exists():
            return candidate
        nested = candidate / "ppe-detection" / "v2_pipeline"
        if nested.exists() and (nested / "src").exists():
            return nested
    raise RuntimeError("Could not find v2_pipeline root")


V2_ROOT = find_v2_root(Path.cwd().resolve())
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from src.dataset.build_experiment_sets import (
    build_ablation_datasets,
    build_class_distribution,
    summarize_experiments,
    verify_experiment_integrity,
)

V2_ROOT

WindowsPath('C:/Github/smart-factory-safety-monitoring-system/ppe-detection/v2_pipeline')

This cell keeps the notebook portable across machines. All heavy file-copy and validation logic stays in `src/dataset/build_experiment_sets.py`, while the notebook focuses on orchestration and review tables.


## Load Configuration

Load dataset locations, class names, and augmentation settings from YAML files.


In [2]:
dataset_config_path = V2_ROOT / "configs" / "dataset_config.yaml"
class_names_path = V2_ROOT / "configs" / "class_names.yaml"
augmentation_config_path = V2_ROOT / "configs" / "augmentation_config.yaml"

with dataset_config_path.open("r", encoding="utf-8") as file_handle:
    dataset_config = yaml.safe_load(file_handle)
with class_names_path.open("r", encoding="utf-8") as file_handle:
    class_config = yaml.safe_load(file_handle)
with augmentation_config_path.open("r", encoding="utf-8") as file_handle:
    augmentation_config = yaml.safe_load(file_handle)

class_names = {int(class_id): name for class_id, name in class_config["names"].items()}
splits_original_dir = V2_ROOT / dataset_config["splits_original_dir"]
augmented_train_dir = V2_ROOT / dataset_config["augmented_train_dir"]
experiments_dir = V2_ROOT / "data" / "experiments"

config_summary = pd.DataFrame(
    [
        {"setting": "splits_original_dir", "value": str(splits_original_dir)},
        {"setting": "augmented_train_dir", "value": str(augmented_train_dir)},
        {"setting": "experiments_dir", "value": str(experiments_dir)},
        {"setting": "classes", "value": class_names},
        {
            "setting": "offline_augmentation",
            "value": augmentation_config["offline_augmentation"],
        },
    ]
)
display(config_summary)

,setting,value
0,splits_original_dir,C:\Github\smart-factory-safety-monitoring-syst...
1,augmented_train_dir,C:\Github\smart-factory-safety-monitoring-syst...
2,experiments_dir,C:\Github\smart-factory-safety-monitoring-syst...
3,classes,"{0: 'person', 1: 'helmet', 2: 'vest'}"
4,offline_augmentation,"{'ir_ratio': 0.25, 'sunlight_ratio': 0.15, 'bl..."


The dataset builder uses these paths without hardcoded absolute locations. The class mapping is also reused when writing Ultralytics dataset YAML files and class-distribution reports.


## Current Original Split Counts

Review the original train, validation, and test split sizes before creating experiments.


In [3]:
valid_image_extensions = {".jpg", ".jpeg", ".png"}


def count_files(directory: Path, suffixes: set[str]) -> int:
    if not directory.exists():
        return 0
    return sum(
        1
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in suffixes
    )


split_counts = pd.DataFrame(
    [
        {
            "split": split,
            "num_images": count_files(
                splits_original_dir / split / "images", valid_image_extensions
            ),
            "num_labels": count_files(splits_original_dir / split / "labels", {".txt"}),
        }
        for split in ["train", "val", "test"]
    ]
)
display(split_counts)

,split,num_images,num_labels
0,train,380,380
1,val,95,95
2,test,50,50


These counts should come directly from Notebook 03. If any split is missing or empty, the build step will stop with a clear error rather than producing incomplete experiment folders.


## Offline Augmentation Counts

Review how many generated training samples are available from Notebook 04.


In [4]:
offline_sources = ["ir", "sunlight", "blur_compression"]
offline_counts = pd.DataFrame(
    [
        {
            "source": source,
            "num_images": count_files(
                augmented_train_dir / source / "images", valid_image_extensions
            ),
            "num_labels": count_files(
                augmented_train_dir / source / "labels", {".txt"}
            ),
        }
        for source in offline_sources
    ]
)
display(offline_counts)

if offline_counts["num_images"].sum() == 0:
    print(
        "Warning: no offline augmented images found. Experiments C and D will contain original train samples only."
    )

,source,num_images,num_labels
0,ir,95,95
1,sunlight,57,57
2,blur_compression,38,38


Offline augmented samples are added only to Experiments C and D. Experiments A and B intentionally use original training images only so the effect of online augmentation can be studied separately later.


## Build Ablation Datasets

Copy original and offline augmented files into the four experiment folders.


In [5]:
overwrite_existing_experiments = True

ablation_report = build_ablation_datasets(
    splits_original_dir=splits_original_dir,
    augmented_train_dir=augmented_train_dir,
    experiments_dir=experiments_dir,
    class_names=class_names,
    overwrite=overwrite_existing_experiments,
)

display(ablation_report.head(20))

,experiment,split,source_type,original_path,copied_path,status,notes
0,exp_A_original_only,train,original,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
1,exp_A_original_only,train,original,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
2,exp_A_original_only,train,original,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
3,exp_A_original_only,train,original,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
4,exp_A_original_only,train,original,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
5,exp_A_original_only,train,original,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
6,exp_A_original_only,train,original,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
7,exp_A_original_only,train,original,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
8,exp_A_original_only,train,original,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
9,exp_A_original_only,train,original,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,


The build function copies files, never moves them. Original train, validation, and test filenames are preserved. If an offline augmented file would conflict with an existing train filename, the builder applies a safe suffix and records a warning.


## Dataset YAML Files

Confirm that each experiment has an Ultralytics dataset YAML file.


In [6]:
dataset_yaml_paths = [
    V2_ROOT / "data_exp_A_original_only.yaml",
    V2_ROOT / "data_exp_B_online_aug.yaml",
    V2_ROOT / "data_exp_C_offline_aug.yaml",
    V2_ROOT / "data_exp_D_full_pipeline.yaml",
]

yaml_status = pd.DataFrame(
    [
        {"yaml_file": path.name, "exists": path.exists(), "path": str(path)}
        for path in dataset_yaml_paths
    ]
)
display(yaml_status)

,yaml_file,exists,path
0,data_exp_A_original_only.yaml,True,C:\Github\smart-factory-safety-monitoring-syst...
1,data_exp_B_online_aug.yaml,True,C:\Github\smart-factory-safety-monitoring-syst...
2,data_exp_C_offline_aug.yaml,True,C:\Github\smart-factory-safety-monitoring-syst...
3,data_exp_D_full_pipeline.yaml,True,C:\Github\smart-factory-safety-monitoring-syst...


These YAML files point Ultralytics to the generated experiment folders. Later notebooks can train or validate by selecting the appropriate YAML for the experiment being run.


## Build Reports

Create summary, class-distribution, and integrity-warning reports for the generated experiments.


In [7]:
summary_df = summarize_experiments(experiments_dir, class_names)
class_distribution_df = build_class_distribution(experiments_dir, class_names)
warnings_df = verify_experiment_integrity(
    experiments_dir=experiments_dir,
    class_names=class_names,
    copy_report=ablation_report,
)

reports_dir = V2_ROOT / "reports" / "experiments"
reports_dir.mkdir(parents=True, exist_ok=True)
ablation_report.to_csv(reports_dir / "ablation_dataset_report.csv", index=False)
summary_df.to_csv(reports_dir / "ablation_dataset_summary.csv", index=False)
class_distribution_df.to_csv(
    reports_dir / "ablation_class_distribution.csv", index=False
)
warnings_df.to_csv(reports_dir / "ablation_integrity_warnings.csv", index=False)

print(f"Saved experiment reports to: {reports_dir}")

Saved experiment reports to: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\experiments


The row-level copy report is an audit trail. The summary report is useful for quick dataset-size checks. The class-distribution report shows whether classes remain represented. The warnings report captures integrity issues that should be reviewed before training.


## Compact Summary Tables

Display the main dataset summaries in notebook output.


In [8]:
display(summary_df)

train_summary = summary_df.loc[summary_df["split"].eq("train")].copy()
display(
    train_summary[["experiment", "num_images", "num_labels", "num_objects", "notes"]]
)

,experiment,split,num_images,num_labels,num_objects,num_person,num_helmet,num_vest,notes
0,exp_A_original_only,train,380,380,2571,1059,660,852,original train only; online augmentation off d...
1,exp_A_original_only,val,95,95,632,249,174,209,fixed split
2,exp_A_original_only,test,50,50,685,271,187,227,fixed split
3,exp_B_online_aug,train,380,380,2571,1059,660,852,original train only; online augmentation on du...
4,exp_B_online_aug,val,95,95,632,249,174,209,fixed split
5,exp_B_online_aug,test,50,50,685,271,187,227,fixed split
6,exp_C_offline_aug,train,570,570,3723,1557,943,1223,original plus offline augmented train; online ...
7,exp_C_offline_aug,val,95,95,632,249,174,209,fixed split
8,exp_C_offline_aug,test,50,50,685,271,187,227,fixed split
9,exp_D_full_pipeline,train,570,570,3723,1557,943,1223,original plus offline augmented train; online ...


,experiment,num_images,num_labels,num_objects,notes
0,exp_A_original_only,380,380,2571,original train only; online augmentation off d...
3,exp_B_online_aug,380,380,2571,original train only; online augmentation on du...
6,exp_C_offline_aug,570,570,3723,original plus offline augmented train; online ...
9,exp_D_full_pipeline,570,570,3723,original plus offline augmented train; online ...


The train summary should show A and B with original train counts only, while C and D should include original plus offline augmented train samples when those augmentations exist.


## Integrity Warnings

Review any warnings before moving into training notebooks.


In [9]:
if warnings_df.empty:
    print("No integrity warnings found.")
else:
    display(warnings_df)


No integrity warnings found.


Some warnings, such as a class missing from a very small split, may be expected for tiny datasets. Warnings about mismatched labels, missing images, or validation/test differences should be fixed before training.


## Checklist Before Notebook 06

- Four experiment folders exist under `data/experiments`.
- Experiments A and B use original train data only.
- Experiments C and D include offline augmented train data if available.
- Validation and test splits are identical across all experiments.
- Dataset YAML files exist for all four experiments.
- CSV reports were saved under `reports/experiments`.
- Any integrity warnings have been reviewed.
